In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 2*ω_a 

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b_0
ω_p=2ω_a-ω_d

ϵ_0 = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=19.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
#ATS + Mean_Field + ladder_operator
function equation_stark!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_0)
    coupling = cos((a+conj(a))/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (b+conj(b))/sqrt(2) * sqrt(Z_b)/ϕ_quantum) / ϕ_quantum / sqrt(2)
        
    du[1] = -1im*(ω_a*a + sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end

#ATS + Mean_Field
function equation_1ODE_stark!(du,u,p::SimulationParams{SystemATS_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    ϕ_a = u[1]
    ϕ_b = u[2]
    ϕ_p_a = u[3]
    ϕ_p_b = u[4]

    coupling = cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum) / ϕ_quantum
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)
        
    du[1] = ϕ_p_a
    du[2] = ϕ_p_b
    du[3] = -ω_a*(ω_a*ϕ_a - sqrt(Z_a) * sin_pump*coupling) - 2 * p.environment_prepared.κ_a * du[1]
    du[4] = -ω_b*(ω_b*ϕ_b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling) - 2 * p.environment_prepared.κ_b * du[2]
    return nothing
end

In [ ]:
function find_drive_freq(F, freqs)
    mask = (freqs .> 7) .& (freqs .< 8.5)
    F = F[mask]
    freqs = freqs[mask]
    freq_max = abs(freqs[argmax(abs.(F))])
    threshold = 0.05 #GHz
    F[argmax(abs.(F))] = 0
    freq_max_bis = abs(freqs[argmax(abs.(F))])
    while abs(freq_max_bis-freq_max)<threshold
        F[argmax(abs.(F))] = 0
        freq_max_bis = abs(freqs[argmax(abs.(F))])
    end
    return(freq_max_bis)
end

In [ ]:
Drive_freq = Dict{Float64,Dict{Float64,Float64}}()
epsilon_d_over_g2_list = range(start = 7, stop = 20, length = 14)
epsilon_p_list = range(start = 0.3, stop = .9, length = 14)
t_max=400/κ_b
a_0 = 0.0
b_0 = 0.0
ω_d = ω_b_0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment)
u_0_tot = get_init(system_prepared, environment_prepared)
for epsilon_p in ProgressBar(epsilon_p_list)
    g2 = besselj(1,epsilon_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    Drive_freq[epsilon_p] = Dict{Float64,Float64}()
    for epsilon_d_over_g2 in epsilon_d_over_g2_list
        ϵ_d = epsilon_d_over_g2 * g2
        drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, epsilon_p, ω_p, pump_phase)
        drive_prepared = prepare(drive)
        dt = 1/20
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        _, b, t_list = default_solver(u_0_tot, (0.0, t_max), p, dt; equation=equation_stark!)
        F, freqs = FFT_perso(real.(b), t_list)
        f_d = find_drive_freq(F, freqs)
        Drive_freq[epsilon_p][epsilon_d_over_g2] = f_d
    end
end

# Check chaos for Mean-field

In [ ]:
t_max = 10/κ_b
re_a_0, im_a_0 = 0, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

system_prepared = prepare(system)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
u_0_new = vcat(u_0,u_dot_0);

In [ ]:
Lyap_coef_chaos = Array{Float64}(undef, length(epsilon_d_over_g2_list), length(epsilon_p_list))
drive = DriveTwoModesBasics(0.0, ω_d, buffer_phase, 0.0, ω_p, pump_phase)
drive_prepared = prepare(drive)
t_w = 30/κ_b
for i in ProgressBar(1:length(epsilon_p_list))
    ϵ_0_i = epsilon_p_list[i]
    drive.ϵ_0 = ϵ_0_i
    g2_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b # (Leon's paper)
    for j in 1:length(epsilon_d_over_g2_list)
        ϵ_d_j = epsilon_d_over_g2_list[j]*g2_i
        drive.ϵ_d = ϵ_d_j
        f_d = Drive_freq[epsilon_p_list[i]][epsilon_d_over_g2_list[j]]
        drive.ω_d = f_d*2*pi
        drive_prepared.drive = drive
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE_stark!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end

In [ ]:
g2_list = E_J.*besselj.(1,epsilon_p_list).*φ_a^2 .*φ_b
α_sq= (epsilon_d_over_g2_list./2)
heatmap(g2_list/2/pi*1e3,α_sq,Lyap_coef_chaos)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(α_sq),maximum(α_sq)),
    colorbar_title="\nLyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(round(κ_b*t_max))"*", "*L"κ_b t_W="*"$(round(κ_b*t_w))",
    clims=(minimum(Lyap_coef_chaos),maximum(Lyap_coef_chaos)),
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);

In [ ]:
heatmap(g2_list/2/pi*1e3,α_sq, Lyap_coef_chaos_sign, cmap=:bluesreds)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"|\alpha_{th}|^2",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(α_sq),maximum(α_sq)),
    colorbar_title="\nSign of Lyapunov exp.", title = "Mean field "*L"κ_b t= "*"$(κ_b*t_max)",
    clims=(minimum(Lyap_coef_chaos_sign),maximum(Lyap_coef_chaos_sign)),
    right_margin=10mm)
#savefig("../../figures/chaos/heatmap_lyapunov_coef_sign_omega_p_0.pdf")